# **Portfolio:** Mario Casanova — Data Science & Analytics Portfolio
## **Case Study:** Telemetry, Stickiness & the Limits of Churn Prediction

---
*This notebook asks a falsifiable question and reports an honest answer, whichever it turns out to be: can mobile-app telemetry predict subscription churn? I engineer a per-user engagement metric, fit an interpretable logistic model, and read the result without flattering it. The answer is **negative** (the telemetry barely beats a coin flip), and the harder, more useful work is understanding why: a device-type confounder, and a temporal misalignment between when engagement is measured and when churn actually happens. Reading a null result correctly is worth more than manufacturing a positive one, and that is the standard this notebook holds itself to.*

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette("magma")
np.random.seed(42)

### 1. Telemetry log ingestion (March 2023)
I load the transactional log that records, day by day, when a user turned on, viewed, or interacted with their device (IoT) through the application.

In [2]:
# Fact Table (Fine Granularity)
df_telemetry = pd.read_csv('../data/telemetry_logs_202303.csv')

# Master Tables to enrich features
df_subs = pd.read_csv('../data/subscriptions.csv')
df_subs['SubscriptionEnd'] = pd.to_datetime(df_subs['SubscriptionEnd'])

# Critical Labeling: Did the user Churn during or shortly after the evaluated telemetry period? (Until May 2023)
df_subs['Has_Churned'] = df_subs['SubscriptionEnd'].apply(lambda x: 1 if pd.notnull(x) and x < pd.Timestamp('2023-06-01') else 0)

print(f"[*] Ingested Raw Telemetry Logs: {len(df_telemetry)} Records.\n")
df_telemetry.head()

[*] Ingested Raw Telemetry Logs: 50126 Records.



,UserID,LogDate,AppOpens,MinutesActive
0,10000,2023-03-24,5,16.4
1,10000,2023-03-27,6,20.8
2,10000,2023-03-05,8,41.2
3,10000,2023-03-09,6,12.7
4,10000,2023-03-13,3,34.4


### 2. Feature engineering: per-user active-day share
A true *stickiness ratio* (DAU/MAU) is a population-level metric, daily actives divided by monthly actives across the whole fleet. What I compute here is its per-user analogue: the share of days in the month a user was active (`Days_Active / 31`). The two are related but not the same construct, and I name it accordingly rather than let the distinction blur. I then consolidate the transactional telemetry into one feature row per user.

In [3]:
# Calculation of aggregated metrics per user (MAU Level)
df_user_features = df_telemetry.groupby('UserID').agg(
    Days_Active=('LogDate', 'count'), # Each transactional row is a unique registration day
    Total_App_Opens=('AppOpens', 'sum'),
    Avg_Minutes_Active_Per_Day=('MinutesActive', 'mean')
).reset_index()

# Individual Stickiness: What percentage of the month (31 days in March) did they use the app?
df_user_features['Stickiness_Ratio'] = (df_user_features['Days_Active'] / 31.0) * 100

# Merge with the Target (Has_Churned)
df_model = pd.merge(df_user_features, df_subs[['UserID', 'Has_Churned']], on='UserID', how='inner')

df_model.head()

,UserID,Days_Active,Total_App_Opens,Avg_Minutes_Active_Per_Day,Stickiness_Ratio,Has_Churned
0,10000,25,147,21.100000,80.645161,1
1,10002,15,67,23.826667,48.387097,1
2,10008,13,49,26.961538,41.935484,1
3,10014,20,91,20.010000,64.516129,0
4,10015,21,90,23.666667,67.741935,0


### 3. Predictive model (logistic regression for churn propensity)
I train a deliberately interpretable model (a plain logit, no black box) on the telemetry features. The question worth asking is not whether the model fits the training data, but whether there is a **usable, causal** relationship of the form "if stickiness falls below X, the customer cancels." The honest answer turns out to be no, and the reason matters methodologically: any apparent link runs through an unobserved common cause, so what looks like a lever is correlation wearing a lever's clothes.

In [4]:
features = ['Stickiness_Ratio', 'Total_App_Opens', 'Avg_Minutes_Active_Per_Day']
X = df_model[features]
y = df_model['Has_Churned']

# 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(class_weight='balanced')
model.fit(X_train, y_train)

y_preds = model.predict(X_test)
y_probs = model.predict_proba(X_test)[:, 1]

print("[*] Classification Report on Test Set:\n")
print(classification_report(y_test, y_preds))
print(f"[*] AUC-ROC Score: {roc_auc_score(y_test, y_probs):.3f}")

[*] Classification Report on Test Set:

              precision    recall  f1-score   support

           0       0.68      0.61      0.65       135
           1       0.48      0.55      0.51        87

    accuracy                           0.59       222
   macro avg       0.58      0.58      0.58       222
weighted avg       0.60      0.59      0.59       222

[*] AUC-ROC Score: 0.562


### 4. Model interpretation, and a caution
The logistic fit below *looks* like a decision boundary, but its slope is shallow and its metrics are weak (AUC 0.562, churner precision 0.48). A plausible-looking curve is not a usable signal — the annotation a careless analyst would reach for here ("danger zone, launch a campaign") is exactly the premature operational story this notebook exists to resist. That is why I'd hold off on any retention trigger until the next two cells, which show why the curve is lying.

In [5]:
fig, ax = plt.subplots(figsize=(10, 6))
# Data plus the logistic fit — note how shallow the fitted curve is.
sns.regplot(data=df_model, x='Stickiness_Ratio', y='Has_Churned', 
            logistic=True, n_boot=50, y_jitter=.05, 
            scatter_kws={'alpha': 0.1, 'color': 'cyan'}, 
            line_kws={'color': 'yellow', 'linewidth': 3}, ax=ax)

ax.set_title("Churn vs Stickiness — a near-flat logistic fit", fontsize=15, pad=15)
ax.set_ylabel("Probability P(Churn = 1)", fontsize=12)
ax.set_xlabel("Stickiness Ratio (% Days of Active Month)", fontsize=12)
ax.set_yticks([0.0, 0.25, 0.5, 0.75, 1.0])

plt.annotate('Shallow slope — weak separation (AUC 0.562)', xy=(20, 0.5), xytext=(35, 0.85),
             arrowprops=dict(facecolor='gray', shrink=0.05), fontsize=10, bbox=dict(facecolor='black', alpha=0.5))

plt.show()

### 5. Post-mortem: why the signal is weak
Two structural problems sit under this weak signal, and neither is a tuning issue. The first is confounding: device type is a common cause of both engagement and churn, and it never enters the model. The second is temporal misalignment: the single telemetry month, March 2023, post-dates the cancellation of most churners, so their features describe behaviour *after* the outcome, not before it. The next two cells show each one directly.

In [6]:
# (1) Confounding: bring DeviceType back in. It drives BOTH engagement and churn,
#     yet it is absent from the feature set — so Stickiness is only its shadow.
df_hw = pd.read_csv('../data/hardware_users.csv')
df_conf = df_model.merge(df_hw[['UserID', 'DeviceType']], on='UserID', how='left')
df_conf.groupby('DeviceType').agg(
    Users=('UserID', 'size'),
    Mean_Stickiness=('Stickiness_Ratio', 'mean'),
    Churn_Rate=('Has_Churned', 'mean'),
).round(2)

,Users,Mean_Stickiness,Churn_Rate
DeviceType,,,
Security Camera,606,62.72,0.31
Smart Doorbell,503,24.75,0.50


In [7]:
# (2) Temporal misalignment: when did churners actually cancel, relative to the
#     March 2023 telemetry window the features are built from?
ended = df_subs.loc[df_subs['Has_Churned'] == 1, 'SubscriptionEnd']
print(f"Churners who cancelled BEFORE the telemetry month (Mar 2023): "
      f"{(ended < pd.Timestamp('2023-03-01')).mean():.1%}")
print(f"                                during March 2023:            "
      f"{((ended >= pd.Timestamp('2023-03-01')) & (ended < pd.Timestamp('2023-04-01'))).mean():.1%}")
print(f"                                April-May 2023:               "
      f"{(ended >= pd.Timestamp('2023-04-01')).mean():.1%}")
print("\nFor most positives the telemetry describes POST-churn behaviour — "
      "a leading indicator must be measured strictly before the outcome.")

Churners who cancelled BEFORE the telemetry month (Mar 2023): 79.6%
                                during March 2023:            9.8%
                                April-May 2023:               10.6%

For most positives the telemetry describes POST-churn behaviour — a leading indicator must be measured strictly before the outcome.


### Conclusion — an honest null
The model does not work, and that is the finding. On the held-out test set it scores AUC 0.562 (a coin flip sits at 0.50) with 0.48 precision on the churner class, so more than half of the flagged users are false alarms. It is not deployable, and the instinct to wire it straight into a CRM pipeline would have shipped noise dressed up as insight.

The value here is in reading the null correctly. This weakness is not cosmetic, the kind a better solver or an extra feature would fix; it is structural, built into how the data was assembled before any model ever saw it. Two things drive it. Device type is a common cause of both engagement and churn (cameras run high stickiness and low churn, doorbells the reverse), and it never enters the model, so the stickiness–churn correlation is only the shadow that confounder casts; pulling the stickiness lever directly would change nothing. And the single telemetry month post-dates most churners' cancellations (about 80% cancelled before the March window), so for the majority of positives the features describe behaviour *after* the outcome, not before it — a leading indicator measured after the event it was supposed to lead.

The correct next step, then, is not a retention campaign but a redesigned study: device type entered as a feature or control, and a strict observation-to-outcome temporal split with telemetry captured before the churn window begins. A negative result, reported honestly and diagnosed to its causes, is the deliverable.